# Análise de Cotas de EEB, LR e Escoamento das Bacias

Este script processa arquivos geográficos para compilar dados das bacias hidrográficas e alternativas de linhas de recalque.

Entrada esperada:
- bacias
- coluna_bacias: nome da coluna do shape de bacias onde tem o nome das bacias
- eeb
- MDE.tif
- caminho_lr: linha de recalque

Saída (definido pelo usuário):
- Para cada bacia: cota da EEB, maior cota ao longo de cada linha de recalque, extensão da linha, e bacia de destino da linha.

O script associa automaticamente os dados às bacias e usa o MDE (com fallback online) para calcular elevações.

In [6]:
import numpy as np
import geopandas as gpd
import rasterio
import requests
from shapely.geometry import Point, LineString
from shapely.geometry.multilinestring import MultiLineString
from geopandas.tools import sjoin
import os

## Funções Auxiliares

In [42]:
def buscar_cota_online(x, y):
    """
    Consulta a API do OpenTopodata para obter a cota (elevação) de um ponto (x, y)
    quando os dados não estão disponíveis no MDE local.

    Retorna:
        float: valor da elevação em metros ou np.nan se a consulta falhar.
    """
    try:
        response = requests.get(f"https://api.opentopodata.org/v1/srtm90m?locations={y},{x}")
        if response.status_code == 200:
            results = response.json().get('results')
            if results and results[0].get('elevation') is not None:
                return results[0]['elevation']
    except Exception:
        pass
    return np.nan

def extrair_cota_com_fallback(ponto, src):
    """
    Extrai a cota (elevação) de um ponto a partir do MDE raster.
    Se não houver valor válido no MDE ou ocorrer erro, tenta obter a cota via API online.

    Argumentos:
        ponto (shapely.geometry.Point ou MultiPoint): ponto a ser avaliado.
        src (rasterio.DatasetReader): MDE aberto para leitura.

    Retorna:
        float: cota extraída ou np.nan em caso de falha.
    """
    if ponto.is_empty:
        return np.nan

    try:
        if ponto.geom_type == 'MultiPoint':
            x, y = ponto.geoms[0].x, ponto.geoms[0].y
        elif ponto.geom_type == 'Point':
            x, y = ponto.x, ponto.y
        else:
            return np.nan  # Tipo não suportado

        row, col = src.index(x, y)
        valor = src.read(1, window=((row, row+1), (col, col+1)))[0, 0]

        if valor == src.nodata or np.isnan(valor):
            return buscar_cota_online(x, y)
        return valor

    except Exception:
        # Se deu erro antes de x, y, tenta recuperar de novo se possível
        try:
            if ponto.geom_type == 'MultiPoint':
                x, y = ponto.geoms[0].x, ponto.geoms[0].y
            elif ponto.geom_type == 'Point':
                x, y = ponto.x, ponto.y
            else:
                return np.nan
            return buscar_cota_online(x, y)
        except Exception:
            return np.nan


from shapely.geometry import LineString, Point

def interpolar_linha_em_pontos(geom, n_amostras=1000):
    """
    Interpola pontos ao longo de uma geometria do tipo LineString ou MultiLineString.
    Converte MultiLineString em LineString contínua e gera uma amostragem uniforme.

    Argumentos:
        geom (shapely.geometry.LineString ou MultiLineString): geometria da linha.
        n_amostras (int): número mínimo de amostras ao longo da linha.

    Retorna:
        list[Point]: lista de pontos interpolados ao longo da linha.
    """
    if geom is None or geom.is_empty:
        return []

    if geom.geom_type == 'MultiLineString':
        coords = []
        for line in geom.geoms:  # ✅ corrigido
            coords.extend(line.coords)
        geom = LineString(coords)

    if geom.geom_type != 'LineString':
        return []

    n = max(n_amostras, int(geom.length))
    return [geom.interpolate(i / n, normalized=True) for i in range(n + 1)]


def extrair_atributos_bacia(geometria, bacias, nome):
    """
    Identifica a bacia que contém a geometria fornecida e retorna seus atributos.

    - Para geometrias lineares, considera o ponto inicial da linha.
    - Para pontos ou multipontos, usa diretamente a geometria.

    Argumentos:
        geometria (shapely geometry): ponto, linha ou multilinha a ser avaliada.
        bacias (GeoDataFrame): polígonos de bacia com campos 'nome' e 'etapa'.

    Retorna:
        Retorna:
        str: valor da coluna indicada por `nome`, ou None se não encontrado.
    """
    if geometria is None or geometria.is_empty:
        return None, None
    if geometria.geom_type in ['LineString', 'MultiLineString']:
        ponto = Point(geometria.coords[0]) if geometria.geom_type == 'LineString' else Point(geometria.geoms[0].coords[0])
    else:
        ponto = geometria
    for _, bacia in bacias.iterrows():
        if ponto.within(bacia.geometry):
            return bacia[nome]
    return None, None

def carregar_shapefile_com_bacia(caminho, bacias,nome):
    """
    Lê um shapefile de pontos ou linhas e associa cada geometria à bacia correspondente.

    Adiciona duas colunas ao GeoDataFrame:
    - 'nome_bacia': nome da bacia onde a geometria está inserida.
    - 'etapa': etapa correspondente à bacia.

    Argumentos:
        caminho (str): caminho do shapefile a ser carregado.
        bacias (GeoDataFrame): bacias poligonais de referência.

    Retorna:
        GeoDataFrame: shapefile com colunas adicionais de bacia.
    """
    gdf = gpd.read_file(caminho)
    gdf['nome_bacia'] = gdf['geometry'].apply(lambda geom: extrair_atributos_bacia(geom, bacias, nome))
    return gdf

def maior_cota_lr(lrs, mde_path):
    """
    Calcula a maior cota ao longo de cada linha de recalque (LR) com base no MDE.
    Se valores do MDE estiverem ausentes, usa API online como fallback.

    Para cada LR:
    - Interpola pontos ao longo da linha.
    - Extrai cota em cada ponto.
    - Identifica a maior cota e o ponto correspondente.

    Argumentos:
        lrs (GeoDataFrame): linhas de recalque com coluna 'nome_bacia'.
        mde_path (str): caminho para o arquivo raster (MDE).

    Retorna:
        GeoDataFrame: com colunas 'bacia_orig', 'cota' e 'geometry' do ponto de maior elevação.
    """
    with rasterio.open(mde_path) as src:
        resultados = []
        for _, lr in lrs.iterrows():
            geom = lr.geometry
            pontos = interpolar_linha_em_pontos(geom)
            cotas = [extrair_cota_com_fallback(p, src) for p in pontos]
            if all(np.isnan(cotas)):
                max_cota = np.nan
                max_ponto = None
            else:
                max_idx = np.nanargmax(cotas)
                max_cota = cotas[max_idx]
                max_ponto = pontos[max_idx]
            resultados.append({
                'bacia_orig': lr['nome_bacia'],
                'cota': max_cota,
                'geometry': Point(max_ponto) if max_ponto else None
            })
        return gpd.GeoDataFrame(resultados, crs=lrs.crs)

def adicionar_cota(gdf, mde_path):
    """
    Adiciona uma coluna 'cota' a um GeoDataFrame de pontos, com base em um MDE.

    Se a cota não estiver disponível no MDE, faz consulta online como fallback.

    Argumentos:
        gdf (GeoDataFrame): pontos (como EEBs) para os quais será atribuída a elevação.
        mde_path (str): caminho do MDE raster.

    Retorna:
        GeoDataFrame: com coluna 'cota' preenchida.
    """
    with rasterio.open(mde_path) as src:
        gdf['cota'] = gdf['geometry'].apply(lambda geom: extrair_cota_com_fallback(geom, src))
    return gdf

def processar_alternativa(df, bacias, lr, pl, sufixo,nome):
    """
    Processa uma alternativa de linha de recalque e integra os dados ao DataFrame principal.

    Para cada bacia de origem:
    - Associa extensão da linha.
    - Cota máxima no trajeto da linha.
    - Bacia de destino (com base no ponto final da linha).

    Argumentos:
        df (DataFrame): tabela principal com as bacias (coluna 'nome').
        bacias (GeoDataFrame): polígonos de bacia.
        lr (GeoDataFrame): linhas de recalque da alternativa.
        pl (GeoDataFrame): pontos de maior cota por LR.
        sufixo (str): identificador da alternativa (ex: 'A1').

    Retorna:
        DataFrame: df atualizado com colunas da alternativa adicionadas.
    """
    def obter_ultimo_ponto(geom):
        if geom is None or geom.is_empty:
            return None
        if geom.geom_type == 'LineString':
            return Point(geom.coords[-1])
        elif geom.geom_type == 'MultiLineString':
            ultima_linha = geom.geoms[-1]
            return Point(ultima_linha.coords[-1])
        else:
            return None

    ultimos_pontos = [obter_ultimo_ponto(geom) for geom in lr.geometry]
    ultimos_gdf = gpd.GeoDataFrame({'bacia_orig': lr['nome_bacia']}, geometry=ultimos_pontos, crs=lr.crs)

    ultimos_com_bacia = sjoin(ultimos_gdf, bacias, how='left', predicate='within')
    ultimos_com_bacia.rename(columns={nome: f'bacia_destino_{sufixo}'}, inplace=True)

    lr['extensao_m'] = lr.geometry.length

    temp_df = pl[['bacia_orig', 'cota']].copy()
    temp_df = temp_df.merge(
        lr[['nome_bacia', 'extensao_m']],
        left_on='bacia_orig', right_on='nome_bacia', how='left'
    ).drop(columns='nome_bacia')

    temp_df = temp_df.merge(
        ultimos_com_bacia[['bacia_orig', f'bacia_destino_{sufixo}']],
        on='bacia_orig', how='left'
    )

    temp_df.columns = ['bacia_orig', f'cota_pl_{sufixo}', f'extensao_lr_{sufixo}', f'bacia_destino_{sufixo}']
    df = df.merge(temp_df, left_on=nome, right_on='bacia_orig', how='left')
    return df.drop(columns=['bacia_orig'])


## Código Principal

In [32]:
# Carrega dados principais
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\Shapes Criados\Projeto2024\BaciasHidro_V02.gpkg')
coluna_bacias = 'Nome'
eeb = carregar_shapefile_com_bacia(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\Shapes Criados\Projeto2024\EEBs.gpkg',bacias,coluna_bacias)
mde_path = os.path.join(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\Aerolevantamento\Fase 5 - MDS, MDT e Curvas de Nível\Mdt_Tapes.tif')
caminho_lr = os.path.join(r'C:\Users\gabriel.coimbra\Downloads\LR_Gabe.gpkg')
saida = r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes'

In [44]:
# Define o nome da alternativa e monta o caminho da linha de recalque
alt_nome = input("Digite o nome da alternativa (ex: A1): ").strip()

if not os.path.exists(caminho_lr):
    print(f"Arquivo de linha de recalque não encontrado: {caminho_lr}")

# DataFrame base com bacias
df = bacias[['Nome']].copy()

# Cotas das EEBs
eeb = adicionar_cota(eeb, mde_path)
eeb_por_bacia = eeb[['nome_bacia', 'cota']].drop_duplicates().rename(columns={'cota': 'cota_eeb'})
df = df.merge(eeb_por_bacia, left_on='Nome', right_on='nome_bacia', how='left').drop(columns='nome_bacia')

# Processa linha de recalque
lr = carregar_shapefile_com_bacia(caminho_lr, bacias,coluna_bacias)
pl = maior_cota_lr(lr, mde_path)
df = processar_alternativa(df, bacias, lr, pl, alt_nome,coluna_bacias)

# Salva CSV
df.to_csv(
    os.path.join(saida,f'bacia_destino_{alt_nome}.csv'),
    index=False,
    sep=';',
    decimal=','
)

print("Planilha gerada com sucesso!")

Digite o nome da alternativa (ex: A1):  a1


Planilha gerada com sucesso!
